In [ ]:
#left join: common matching records from both tables + unmatching records from left table
# left anti , let semi - avoid right table
# left anti - unmatching records from left table
# left semi - matching records from left table

In [ ]:

from pyspark.sql.types import * # import the all data types
from pyspark.sql.functions import * # import the all


data = [
    (1, "tiru",    "srikanth", "delhi",     "2019-01-01", None, 1),
    (2, "rama",    "srinu",    "hyderabad", "2019-02-01", None, 1),
    (3, "krishna", "mahesh",   "delhi",     "2019-03-01", None, 1),
    (4, "Sai",     "Thomas",   "pune",      "2019-04-01", None, 1)
] # historical data

schema = StructType([
    StructField("Cust_ID", IntegerType(), True),
    StructField("Cust_Name", StringType(), True),
    StructField("manager", StringType(), True),
    StructField("city", StringType(), True),
    StructField("Start_date", StringType(), True),
    StructField("end_date", StringType(), True),
    StructField("is_active", IntegerType(), True)
]) # here define the own shema defination

df_hist = spark.createDataFrame(data, schema) \
    .withColumn("Start_date", to_date("Start_date")) \
    .withColumn("end_date", to_date("end_date"))

df_hist.show()


In [ ]:
data = [
    (2, "rama", "srinu", "Bangalore"),
    (5, "siva", "koti",  "Mumbai")
]

schema = StructType([
    StructField("Cust_ID", IntegerType(), True),
    StructField("Cust_Name", StringType(), True),
    StructField("manager", StringType(), True),
    StructField("city", StringType(), True)
])

df_inc = spark.createDataFrame(data, schema)

df_inc.show()


In [ ]:
df_old_match=df_hist.alias('hist').join(df_inc.alias('inc'),col("hist.Cust_ID")==col("inc.Cust_ID"),"leftsemi").withColumn('end_date', current_date()).withColumn('is_active',lit("0"))
df_old_match.show()
# historical lookig old record but expeting updates from incremental load

In [ ]:
df_update = (
    df_inc.alias("inc")
    .join(
        df_hist.alias("hist"),
        col("hist.Cust_ID") == col("inc.Cust_ID"),
        "leftsemi"
    )
    .withColumn("start_date", current_date())
    .withColumn("end_date", lit(None).cast(DateType()))  # ✅ FIX
    .withColumn("is_active", lit(1))                      # better as int
)

df_update.show()


In [ ]:
df_old=df_hist.alias('hist').join(df_inc.alias('inc'),col("hist.Cust_ID")==col("inc.Cust_ID"),"left_anti")
df_old.show()


In [ ]:
df_new=df_inc.alias('inc').join(df_hist.alias('hist'),col("hist.Cust_ID")==col("inc.Cust_ID"),"leftanti").withColumn('start_date', current_date()).withColumn("end_date", lit(None).cast(DateType())).withColumn('is_active',lit("1"))
df_new.show()


In [ ]:
from functools import reduce
from pyspark.sql.functions import DataFrame
def unionall(*a):
    return reduce(DataFrame.unionAll,a)
df_final=unionall(df_update,df_old,df_new,df_old_match)
df_final.orderBy("Cust_ID").show()
